# Clear cut method of training the classifier following methodolgy of the thesis

In [1]:
import sys
sys.path.insert(0, "/home/s232534/scripts")

from optimal_percentiles import percentile_calculate
from gtfs_functions import calculate_optimal_p_stops

from segmentation_pelt import trip_classification

import matplotlib.pyplot as plt

import re
import numpy as np
import pandas as pd
from xgboost import XGBClassifier, XGBRFClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from exploration import functions

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import metrics
from multiprocessing import cpu_count

from sklearn.inspection import permutation_importance

## Load the data

In [2]:
df_stats = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_percentile_map = pd.read_parquet("/home/s232534/data/percentile_pipeline/percentiles_071125.parquet")
df_stops_1000m = pd.read_parquet("/home/s232534/data/stops_analysis_071125/stops_statistical_summary_enriched.parquet")
df_stops_250m = pd.read_csv("/home/s232534/data/stops_analysis_250m_050126/stops_statistical_summary_250m_050126.csv")

# Uni-modal ONLY.
df_stats = df_stats.loc[~df_stats['MULTI_MODAL']] 
df_percentile_map = df_percentile_map.loc[~df_percentile_map['MULTI_MODAL']] 
df_stops_1000m = df_stops_1000m.loc[~df_stops_1000m['MULTI_MODAL']] 
df_stops_250m = df_stops_250m.loc[~df_stops_250m['MULTI_MODAL']] 
# ---
# Drop modes we will not be using for this thesis and modes to merge and remap:
# ---
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

df_stats = functions.remap_and_drop_modes_df(df_stats, mode_remap, modes_to_drop)
df_percentile_map = functions.remap_and_drop_modes_df(df_percentile_map, mode_remap, modes_to_drop)
df_stops_1000m = functions.remap_and_drop_modes_df(df_stops_1000m, mode_remap, modes_to_drop)
df_stops_250m = functions.remap_and_drop_modes_df(df_stops_250m, mode_remap, modes_to_drop)

# ---
# Filter speeds
# ---
threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_stats = functions.filter_motion_data_speed_avg(df_stats, threshold)
df_stats = functions.filter_temporal_data(df_stats, time_thres=180)

keep_ids = set(df_stats['TRIP_KEY'])
df_percentile_map = df_percentile_map[df_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_1000m = df_stops_1000m[df_stops_1000m['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_250m = df_stops_250m[df_stops_250m['TRIP_KEY'].isin(keep_ids)].copy()

df_percentile_map = functions.filter_motion_data_speed_percentile(df_percentile_map, threshold, 'SPEED_P85')

keep_ids_p = set(df_percentile_map['TRIP_KEY'])
df_stats = df_stats[df_stats['TRIP_KEY'].isin(keep_ids_p)].copy()
df_stops_1000m = df_stops_1000m[df_stops_1000m['TRIP_KEY'].isin(keep_ids_p)].copy()
df_stops_250m = df_stops_250m[df_stops_250m['TRIP_KEY'].isin(keep_ids_p)].copy()

# Intermediatory validation (should be True True):
print(len(df_stats) == len(df_percentile_map) == len(df_stops_1000m) == len(df_stops_250m))
print(sorted(df_stats['TRIP_KEY']) == sorted(df_percentile_map['TRIP_KEY']) == sorted(df_stops_1000m['TRIP_KEY']) == sorted(df_stops_250m['TRIP_KEY']))

# Here we create our feature dataset that we want to evaluate on the mode, we are fusing our 3 engineered datasets-
# the stops, the motion features and the optimal percentiles.
# We end up with "df_merge" which contains all the information we need to train and evaluate the models.
# Only load the optimal cols
best_p, js_agg, all_js = percentile_calculate.calculate_percentile_distributions(df_pm = df_percentile_map, remap = False, mode = 'mean')
percentile_optimal_list = best_p['col'].tolist()
percentile_optimal_list.append('TRIP_KEY')

df_percentile_optimal = df_percentile_map[percentile_optimal_list]

drop_stat_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_AVG', 'B_MED', 'B_85P', 'B_STD', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_AVG', 'BR_MED', 'BR_85P', 'BR_STD', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'BC_AVG', 'BC_MED', 'BC_85P', 'BC_STD', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84', 'AVG_PT_DIST', 'MED_PT_DIST', 'DIST_TOTAL']
df_stats_slim = df_stats.drop(columns=drop_stat_cols)

# Stops 1000m
best_p_stop_1000m, js_agg_stop_1000m, all_js_stop_1000m = calculate_optimal_p_stops.calculate_percentile_distributions(df_pm = df_stops_1000m, bins=50, mode='mean')
percentile_optimal_list_stop_1000m = best_p_stop_1000m['col'].tolist()
percentile_optimal_list_stop_1000m.append('TRIP_KEY')
df_stops_optimal_1000m = df_stops_1000m[percentile_optimal_list_stop_1000m]

# Stops 250m
best_p_stop_250m, js_agg_stop_250m, all_js_stop_250m = calculate_optimal_p_stops.calculate_percentile_distributions(df_pm = df_stops_250m, bins=50, mode='mean')
percentile_optimal_list_stop_250m = best_p_stop_250m['col'].tolist()
percentile_optimal_list_stop_250m.append('TRIP_KEY')
df_stops_optimal_250m = df_stops_250m[percentile_optimal_list_stop_250m]

# Find al _Pxx stops 1000m
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_1000m.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too 
df_stops_dropped_1000m = df_stops_1000m.drop(columns=drop_cols_Pxx)
df_stops_optimal_1000m = df_stops_optimal_1000m.merge(df_stops_dropped_1000m, on='TRIP_KEY', how='left', validate='one_to_one')

# 250m
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_250m.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too 
df_stops_dropped_250m = df_stops_250m.drop(columns=drop_cols_Pxx)
df_stops_optimal_250m = df_stops_optimal_250m.merge(df_stops_dropped_250m, on='TRIP_KEY', how='left', validate='one_to_one')


# Join the perctile_opt map to the df_stats ->
df_merge = df_stats_slim.merge(df_percentile_optimal, on='TRIP_KEY', how='left', validate='one_to_one')
df_merge = df_merge.merge(df_stops_optimal_1000m, on='TRIP_KEY', how='left', validate='one_to_one')

# Re introduce non-WGS84 in optimal percentiles - bit inefficient but we drop them again:
merge_drop = ['BC_P50', 'BR_P50']
df_merge = df_merge.drop(columns=merge_drop)


# ------ INTRODUCE ROUTE-MATCHED COLS ----
# Load the correct data for map matching:
df_mapmatched = pd.read_csv("/home/s232534/scripts/map_matching/map_matching_interpolated_percentages.csv") 
df_mapmatched = df_mapmatched.drop(columns=['MAIN_MODE', 'MULTI_MODAL'])

# Merge in our new feature cols:
df_merge_matched = df_merge.merge(df_mapmatched, on='TRIP_KEY', how='left', validate='one_to_one')

# Currently non-matched cols are NaN, fill with zero
cols = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_merge_matched[cols] = df_merge_matched[cols].fillna(0)


#----Seperate for merged with 250m instead kinda messy...sorry
df_merge_250m = df_stats_slim.merge(df_percentile_optimal, on='TRIP_KEY', how='left', validate='one_to_one')
df_merge_250m = df_merge_250m.merge(df_stops_optimal_250m, on='TRIP_KEY', how='left', validate='one_to_one')

merge_drop = ['BC_P50', 'BR_P50']
df_merge_250m = df_merge_250m.drop(columns=merge_drop)

df_merge_250m_matched = df_merge_250m.merge(df_mapmatched, on='TRIP_KEY', how='left', validate='one_to_one')
df_merge_250m_matched[cols] = df_merge_250m_matched[cols].fillna(0)

At WALKING threshold 6.5; 83 are above.
At BIKE threshold 10; 7 are above.
At CAR threshold 39; 7 are above.
At BUS threshold 14; 13 are above.
At TRAMWAY threshold 19.5; 0 are above.
At SUBWAY threshold 11.1; 8 are above.
At TRAIN threshold 61; 4 are above.
Filtered by TOTAL_TIME > 180. Dropped 4587 rows.
At WALKING threshold 6.5; 361 are above.
At BIKE threshold 10; 46 are above.
At CAR threshold 39; 10 are above.
At BUS threshold 14; 154 are above.
At TRAMWAY threshold 19.5; 5 are above.
At SUBWAY threshold 11.1; 64 are above.
At TRAIN threshold 61; 16 are above.
True
True
          col   mode_a   mode_b        js
0   SPEED_P50   SUBWAY  WALKING  0.004241
1   SPEED_P50   SUBWAY    TRAIN  0.100848
2   SPEED_P50   SUBWAY      BUS  0.041378
3   SPEED_P50   SUBWAY      CAR  0.396111
4   SPEED_P50   SUBWAY     BIKE  0.321445
5   SPEED_P50   SUBWAY  TRAMWAY  0.049499
6   SPEED_P50  WALKING    TRAIN  0.093467
7   SPEED_P50  WALKING      BUS  0.039190
8   SPEED_P50  WALKING      CAR  0.3990

In [3]:
len(df_merge_matched)

53183

## Distil into stages

In [3]:
best_p_stop_1000m

,col,mean,std,median,min,max,feature
18,STOPS_CLOSEST_BUS_P95,0.107426,0.076436,0.097375,0.012881,0.269164,STOPS_CLOSEST_BUS
37,STOPS_CLOSEST_METRO_P95,0.123158,0.088266,0.085886,0.023663,0.296695,STOPS_CLOSEST_METRO
40,STOPS_CLOSEST_RAIL_P20,0.138146,0.154124,0.054153,0.022549,0.437177,STOPS_CLOSEST_RAIL
58,STOPS_CLOSEST_TRAM_P15,0.139345,0.145755,0.057526,0.019081,0.442003,STOPS_CLOSEST_TRAM
94,STOPS_MINDIST_P95,0.109710,0.080876,0.088380,0.015096,0.262968,STOPS_MINDIST
103,STOPS_NUM_P5,0.142590,0.102955,0.135965,0.016757,0.364572,STOPS_NUM


In [4]:
best_p_stop_250m

,col,mean,std,median,min,max,feature
4,STOPS_CLOSEST_BUS_P30,0.117668,0.082326,0.094581,0.024040,0.305897,STOPS_CLOSEST_BUS
20,STOPS_CLOSEST_METRO_P15,0.093541,0.048340,0.090722,0.030642,0.181175,STOPS_CLOSEST_METRO
41,STOPS_CLOSEST_RAIL_P25,0.127412,0.079587,0.116796,0.032996,0.313393,STOPS_CLOSEST_RAIL
58,STOPS_CLOSEST_TRAM_P15,0.107460,0.083768,0.088700,0.031724,0.334911,STOPS_CLOSEST_TRAM
80,STOPS_MINDIST_P30,0.122622,0.086001,0.096505,0.023555,0.304701,STOPS_MINDIST
103,STOPS_NUM_P5,0.111389,0.078655,0.072720,0.018689,0.271484,STOPS_NUM


In [5]:
df_merge.keys()

Index(['TRIP_KEY', 'MAIN_MODE', 'TOTAL_TIME', 'AVG_PT_DIST_WGS84',
       'MED_PT_DIST_WGS84', 'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG',
       'SPEED_MED', 'SPEED_STD', 'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG',
       'JERK_MED', 'JERK_STD', 'BC_AVG_WGS84', 'BC_MED_WGS84', 'BC_STD_WGS84',
       'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84', 'ACC_P95',
       'BC_WGS84_P50', 'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95',
       'STOPS_CLOSEST_BUS_P95', 'STOPS_CLOSEST_METRO_P95',
       'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95',
       'STOPS_NUM_P5', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN',
       'STOPS_MINDIST_AVG', 'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL',
       'DURATION_BUS', 'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL',
       'MIN_DIST_BUS'],
      dtype='object')

In [6]:
best_p

,col,mean,std,median,min,max,feature
9,ACC_P95,0.172315,0.129013,0.168014,0.012344,0.471362,ACC
10,BC_P50,0.151490,0.114989,0.126688,0.009892,0.428235,BC
20,BC_WGS84_P50,0.151334,0.115023,0.126905,0.009691,0.428339,BC_WGS84
30,BR_P50,0.137003,0.109715,0.100255,0.009707,0.405602,BR
40,BR_WGS84_P50,0.137567,0.109507,0.100551,0.009614,0.405817,BR_WGS84
59,JERK_P95,0.113753,0.077041,0.108414,0.009209,0.289854,JERK
69,SPEED_P95,0.313040,0.145014,0.346963,0.035247,0.556182,SPEED


In [7]:
# For 1000m 
df_simple_motion = df_merge[['TRIP_KEY', 'MAIN_MODE', 'TOTAL_TIME',
       'AVG_PT_DIST_WGS84', 'MED_PT_DIST_WGS84',
       'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG', 'SPEED_MED', 'SPEED_STD',
       'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG', 'JERK_MED', 'JERK_STD' ,'BC_AVG_WGS84', 
       'BC_MED_WGS84', 'BC_STD_WGS84', 'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84']]

df_optimal_motion_percentiles = df_merge[['TRIP_KEY', 'MAIN_MODE', 'ACC_P95', 'BC_WGS84_P50',
       'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95']]

df_stop_proximity = df_merge[['TRIP_KEY', 'MAIN_MODE', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN', 'STOPS_MINDIST_AVG',
       'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL', 'DURATION_BUS',
       'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL', 'MIN_DIST_BUS']]

df_stop_proximity_optimal_percentiles = df_merge[['TRIP_KEY', 'MAIN_MODE', 'STOPS_CLOSEST_BUS_P95',
       'STOPS_CLOSEST_METRO_P95', 'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95', 'STOPS_NUM_P5']]

In [8]:
# Also add the branch for 250m
df_stop_proximity_250m = df_merge_250m[['TRIP_KEY', 'MAIN_MODE', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN', 'STOPS_MINDIST_AVG',
       'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL', 'DURATION_BUS',
       'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL', 'MIN_DIST_BUS']]

df_stop_proximity_optimal_percentiles_250m = df_merge_250m[['TRIP_KEY', 'MAIN_MODE', 'STOPS_CLOSEST_BUS_P30',
       'STOPS_CLOSEST_METRO_P15', 'STOPS_CLOSEST_RAIL_P25', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P30', 'STOPS_NUM_P5']]

# Simple Motion Features

In [9]:
xgb_simple, y_pred_simple, y_test_simple, class_names_simple = trip_classification.run_xgb_classifier(df_simple_motion)

f1_simple = f1_score(y_test_simple, y_pred_simple, average="macro")
print("Macro F1 simple :", f1_simple)

print("\nClassification report for simple:")
print(classification_report(y_test_simple, y_pred_simple, target_names=class_names_simple))

Macro F1 simple : 0.6593707293820658

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.79      0.82       720
         BUS       0.61      0.44      0.51       489
         CAR       0.91      0.91      0.91      4717
      SUBWAY       0.71      0.49      0.58       733
       TRAIN       0.80      0.61      0.69       462
     TRAMWAY       0.59      0.16      0.26        98
     WALKING       0.79      0.93      0.86      3418

    accuracy                           0.84     10637
   macro avg       0.75      0.62      0.66     10637
weighted avg       0.83      0.84      0.83     10637



In [10]:
cm_simple = metrics.confusion_matrix(y_test_simple, y_pred_simple)
df_cm_simple = trip_classification.generate_confusion_table(cm_simple, class_names_simple)
df_cm_simple

Predicted class,BIKE,BUS,CAR,SUBWAY,TRAIN,TRAMWAY,WALKING,Row Sum,Row Recall
Actual class,,,,,,,,,
BIKE,572.000000,3.000000,40.000000,7.000000,0.000000,1.000000,97.000000,720.000000,79.444444
BUS,7.000000,213.000000,140.000000,19.000000,3.000000,4.000000,103.000000,489.000000,43.558282
CAR,26.000000,57.000000,4272.000000,16.000000,48.000000,5.000000,293.000000,4717.000000,90.566038
SUBWAY,23.000000,25.000000,40.000000,362.000000,7.000000,1.000000,275.000000,733.000000,49.386085
TRAIN,3.000000,12.000000,78.000000,46.000000,280.000000,0.000000,43.000000,462.000000,60.606061
TRAMWAY,0.000000,16.000000,31.000000,10.000000,2.000000,16.000000,23.000000,98.000000,16.326531
WALKING,46.000000,23.000000,109.000000,53.000000,10.000000,0.000000,3177.000000,3418.000000,92.949093
Col Sum,677.000000,349.000000,4710.000000,513.000000,350.000000,27.000000,4011.000000,10637.000000,NaN
Col Precision,84.490399,61.031519,90.700637,70.565302,80.000000,59.259259,79.207180,NaN,NaN


In [11]:
latex_table_simple = df_cm_simple.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_simple)

\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 572.000 & 3.000 & 40.000 & 7.000 & 0.000 & 1.000 & 97.000 & 720.000 & 79.444 \\
BUS & 7.000 & 213.000 & 140.000 & 19.000 & 3.000 & 4.000 & 103.000 & 489.000 & 43.558 \\
CAR & 26.000 & 57.000 & 4272.000 & 16.000 & 48.000 & 5.000 & 293.000 & 4717.000 & 90.566 \\
SUBWAY & 23.000 & 25.000 & 40.000 & 362.000 & 7.000 & 1.000 & 275.000 & 733.000 & 49.386 \\
TRAIN & 3.000 & 12.000 & 78.000 & 46.000 & 280.000 & 0.000 & 43.000 & 462.000 & 60.606 \\
TRAMWAY & 0.000 & 16.000 & 31.000 & 10.000 & 2.000 & 16.000 & 23.000 & 98.000 & 16.327 \\
WALKING & 46.000 & 23.000 & 109.000 & 53.000 & 10.000 & 0.000 & 3177.000 & 3418.000 & 92.949 \\
Col Sum & 677.000 & 349.000 & 4710.000 & 513.000 & 350.000 & 27.000 & 4011.000 & 10637.000 & - \\
Col Precision & 84.490 & 61.032 & 90.701 & 70.565 & 80.000 & 59.259 & 79.207 & 

# Simple Motion + Optimal Percentile

In [12]:
df_fixed85_motion_percentiles = df_percentile_map[['TRIP_KEY', 'MAIN_MODE', 'ACC_P85', 'BC_WGS84_P85',
       'BR_WGS84_P85', 'JERK_P85', 'SPEED_P85']]

In [13]:
# 85th PERCENTILE ONLY
temp = df_fixed85_motion_percentiles.copy()
temp = temp.drop(columns="MAIN_MODE")

df_merged_motion = df_simple_motion.merge(temp, on='TRIP_KEY')

xgb_mm85, y_pred_mm85, y_test_mm85, class_names_mm85 = trip_classification.run_xgb_classifier(df_merged_motion)

f1_mm85 = f1_score(y_test_mm85, y_pred_mm85, average="macro")
print("Macro F1 simple :", f1_mm85)

print("\nClassification report for simple:")
print(classification_report(y_test_mm85, y_pred_mm85, target_names=class_names_mm85))

# Table generation:
cm_mm85 = metrics.confusion_matrix(y_test_mm85, y_pred_mm85)
df_cm_mm85 = trip_classification.generate_confusion_table(cm_mm85, class_names_mm85)

print("Combined table with differences")
cm_combined_mm85_df = trip_classification.generate_combined_difference_table_all(cm_mm85, cm_simple, df_cm_mm85, class_names_mm85, df_cm_simple)
#print(cm_combined_mm85_df)

print("\n")
latex_table_diff_mm85 = cm_combined_mm85_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm85)

Macro F1 simple : 0.6705937712528608

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.83      0.80      0.82       720
         BUS       0.60      0.43      0.50       489
         CAR       0.91      0.91      0.91      4717
      SUBWAY       0.70      0.49      0.58       733
       TRAIN       0.81      0.61      0.70       462
     TRAMWAY       0.63      0.22      0.33        98
     WALKING       0.80      0.93      0.86      3418

    accuracy                           0.84     10637
   macro avg       0.75      0.63      0.67     10637
weighted avg       0.83      0.84      0.83     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 575 (3) & 6 (3) & 38 (-2) & 5 (-2) & 1 (1) & 1 (0) & 94 (-3) & 720 (0) & 79.861 (0.417) \\
BUS & 12 (5)

In [14]:
temp = df_optimal_motion_percentiles.copy()
temp = temp.drop(columns="MAIN_MODE")

df_merged_motion = df_simple_motion.merge(temp, on='TRIP_KEY')

xgb_mm, y_pred_mm, y_test_mm, class_names_mm = trip_classification.run_xgb_classifier(df_merged_motion)

f1_mm = f1_score(y_test_mm, y_pred_mm, average="macro")
print("Macro F1 simple :", f1_mm)

print("\nClassification report for simple:")
print(classification_report(y_test_mm, y_pred_mm, target_names=class_names_mm))

# Table generation:
cm_mm = metrics.confusion_matrix(y_test_mm, y_pred_mm)
df_cm_mm = trip_classification.generate_confusion_table(cm_mm, class_names_mm)

print("Combined table with differences")
cm_combined_mm_df = trip_classification.generate_combined_difference_table_all(cm_mm, cm_simple, df_cm_mm, class_names_mm, df_cm_simple)
#print(cm_combined_mm_df)

print("\n")
latex_table_diff_mm = cm_combined_mm_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm)

Macro F1 simple : 0.6655626043943302

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.79      0.81       720
         BUS       0.61      0.43      0.50       489
         CAR       0.90      0.90      0.90      4717
      SUBWAY       0.71      0.50      0.59       733
       TRAIN       0.79      0.61      0.69       462
     TRAMWAY       0.62      0.20      0.31        98
     WALKING       0.79      0.93      0.86      3418

    accuracy                           0.83     10637
   macro avg       0.75      0.62      0.67     10637
weighted avg       0.83      0.83      0.83     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 566 (-6) & 5 (2) & 45 (5) & 6 (-1) & 0 (0) & 0 (-1) & 98 (1) & 720 (0) & 78.611 (-0.833) \\
BUS & 11 (4

## Classification performance of percentiles on its own

### 85th only

In [15]:
# 85th

temp = df_fixed85_motion_percentiles.copy()
#temp = temp.drop(columns="MAIN_MODE")

xgb_mm85, y_pred_mm85, y_test_mm85, class_names_mm85 = trip_classification.run_xgb_classifier(temp)

f1_mm85 = f1_score(y_test_mm85, y_pred_mm85, average="macro")
print("Macro F1 simple :", f1_mm85)

print("\nClassification report for simple:")
print(classification_report(y_test_mm85, y_pred_mm85, target_names=class_names_mm85))

# Table generation:
cm_mm85 = metrics.confusion_matrix(y_test_mm85, y_pred_mm85)
df_cm_mm85 = trip_classification.generate_confusion_table(cm_mm85, class_names_mm85)

Macro F1 simple : 0.4433242094865105

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.62      0.65      0.64       720
         BUS       0.35      0.16      0.21       489
         CAR       0.87      0.87      0.87      4717
      SUBWAY       0.22      0.01      0.02       733
       TRAIN       0.67      0.35      0.46       462
     TRAMWAY       0.23      0.06      0.10        98
     WALKING       0.70      0.95      0.80      3418

    accuracy                           0.76     10637
   macro avg       0.52      0.43      0.44     10637
weighted avg       0.71      0.76      0.72     10637



In [16]:
print(df_cm_mm85.fillna("-").to_latex(index=True, float_format="%.3f"))
df_cm_mm85

\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 466.000 & 29.000 & 99.000 & 2.000 & 0.000 & 5.000 & 119.000 & 720.000 & 64.722 \\
BUS & 59.000 & 76.000 & 214.000 & 2.000 & 3.000 & 3.000 & 132.000 & 489.000 & 15.542 \\
CAR & 102.000 & 46.000 & 4105.000 & 4.000 & 71.000 & 7.000 & 382.000 & 4717.000 & 87.026 \\
SUBWAY & 37.000 & 21.000 & 58.000 & 9.000 & 2.000 & 2.000 & 604.000 & 733.000 & 1.228 \\
TRAIN & 13.000 & 11.000 & 127.000 & 6.000 & 161.000 & 2.000 & 142.000 & 462.000 & 34.848 \\
TRAMWAY & 11.000 & 11.000 & 37.000 & 2.000 & 3.000 & 6.000 & 28.000 & 98.000 & 6.122 \\
WALKING & 58.000 & 25.000 & 76.000 & 16.000 & 0.000 & 1.000 & 3242.000 & 3418.000 & 94.851 \\
Col Sum & 746.000 & 219.000 & 4716.000 & 41.000 & 240.000 & 26.000 & 4649.000 & 10637.000 & - \\
Col Precision & 62.466 & 34.703 & 87.044 & 21.951 & 67.083 & 23.077 & 69.735 & - & -

Predicted class,BIKE,BUS,CAR,SUBWAY,TRAIN,TRAMWAY,WALKING,Row Sum,Row Recall
Actual class,,,,,,,,,
BIKE,466.000000,29.000000,99.000000,2.000000,0.000000,5.000000,119.000000,720.000000,64.722222
BUS,59.000000,76.000000,214.000000,2.000000,3.000000,3.000000,132.000000,489.000000,15.541922
CAR,102.000000,46.000000,4105.000000,4.000000,71.000000,7.000000,382.000000,4717.000000,87.025652
SUBWAY,37.000000,21.000000,58.000000,9.000000,2.000000,2.000000,604.000000,733.000000,1.227831
TRAIN,13.000000,11.000000,127.000000,6.000000,161.000000,2.000000,142.000000,462.000000,34.848485
TRAMWAY,11.000000,11.000000,37.000000,2.000000,3.000000,6.000000,28.000000,98.000000,6.122449
WALKING,58.000000,25.000000,76.000000,16.000000,0.000000,1.000000,3242.000000,3418.000000,94.850790
Col Sum,746.000000,219.000000,4716.000000,41.000000,240.000000,26.000000,4649.000000,10637.000000,NaN
Col Precision,62.466488,34.703196,87.044105,21.951220,67.083333,23.076923,69.735427,NaN,NaN


### Optimal only

In [17]:
xgb_mm, y_pred_mm, y_test_mm, class_names_mm = trip_classification.run_xgb_classifier(df_optimal_motion_percentiles)

f1_mm = f1_score(y_test_mm, y_pred_mm, average="macro")
print("Macro F1 simple :", f1_mm)

print("\nClassification report for simple:")
print(classification_report(y_test_mm, y_pred_mm, target_names=class_names_mm))

# Table generation:
cm_mm = metrics.confusion_matrix(y_test_mm, y_pred_mm)
df_cm_mm = trip_classification.generate_confusion_table(cm_mm, class_names_mm)

Macro F1 simple : 0.5044641326507043

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.73      0.73      0.73       720
         BUS       0.42      0.25      0.31       489
         CAR       0.88      0.88      0.88      4717
      SUBWAY       0.40      0.07      0.12       733
       TRAIN       0.73      0.53      0.61       462
     TRAMWAY       0.21      0.03      0.05        98
     WALKING       0.73      0.94      0.82      3418

    accuracy                           0.78     10637
   macro avg       0.58      0.49      0.50     10637
weighted avg       0.75      0.78      0.75     10637



In [18]:
print(df_cm_mm.fillna("-").to_latex(index=True, float_format="%.3f"))
df_cm_mm

\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 529.000 & 14.000 & 58.000 & 3.000 & 0.000 & 1.000 & 115.000 & 720.000 & 73.472 \\
BUS & 24.000 & 123.000 & 211.000 & 9.000 & 6.000 & 5.000 & 111.000 & 489.000 & 25.153 \\
CAR & 64.000 & 82.000 & 4144.000 & 28.000 & 73.000 & 1.000 & 325.000 & 4717.000 & 87.852 \\
SUBWAY & 36.000 & 32.000 & 67.000 & 53.000 & 4.000 & 0.000 & 541.000 & 733.000 & 7.231 \\
TRAIN & 7.000 & 3.000 & 108.000 & 16.000 & 244.000 & 1.000 & 83.000 & 462.000 & 52.814 \\
TRAMWAY & 3.000 & 22.000 & 35.000 & 10.000 & 3.000 & 3.000 & 22.000 & 98.000 & 3.061 \\
WALKING & 66.000 & 20.000 & 92.000 & 14.000 & 6.000 & 3.000 & 3217.000 & 3418.000 & 94.119 \\
Col Sum & 729.000 & 296.000 & 4715.000 & 133.000 & 336.000 & 14.000 & 4414.000 & 10637.000 & - \\
Col Precision & 72.565 & 41.554 & 87.890 & 39.850 & 72.619 & 21.429 & 72.882 & - & 

Predicted class,BIKE,BUS,CAR,SUBWAY,TRAIN,TRAMWAY,WALKING,Row Sum,Row Recall
Actual class,,,,,,,,,
BIKE,529.000000,14.000000,58.000000,3.000000,0.000000,1.000000,115.000000,720.000000,73.472222
BUS,24.000000,123.000000,211.000000,9.000000,6.000000,5.000000,111.000000,489.000000,25.153374
CAR,64.000000,82.000000,4144.000000,28.000000,73.000000,1.000000,325.000000,4717.000000,87.852449
SUBWAY,36.000000,32.000000,67.000000,53.000000,4.000000,0.000000,541.000000,733.000000,7.230559
TRAIN,7.000000,3.000000,108.000000,16.000000,244.000000,1.000000,83.000000,462.000000,52.813853
TRAMWAY,3.000000,22.000000,35.000000,10.000000,3.000000,3.000000,22.000000,98.000000,3.061224
WALKING,66.000000,20.000000,92.000000,14.000000,6.000000,3.000000,3217.000000,3418.000000,94.119368
Col Sum,729.000000,296.000000,4715.000000,133.000000,336.000000,14.000000,4414.000000,10637.000000,NaN
Col Precision,72.565158,41.554054,87.889714,39.849624,72.619048,21.428571,72.881740,NaN,NaN


# Stop Proximity Features

1000m

In [19]:
temp = df_stop_proximity.copy()
temp = temp.drop(columns="MAIN_MODE")

df_mm_stop = df_merged_motion.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop, y_pred_mm_stop, y_test_mm_stop, class_names_mm_stop = trip_classification.run_xgb_classifier(df_mm_stop)

f1_mm_stop = f1_score(y_test_mm_stop, y_pred_mm_stop, average="macro")
print("Macro F1 simple :", f1_mm_stop)

print("\nClassification report for simple:")
print(classification_report(y_test_mm_stop, y_pred_mm_stop, target_names=class_names_mm_stop))

# Table generation:
cm_mm_stop = metrics.confusion_matrix(y_test_mm_stop, y_pred_mm_stop)
df_cm_mm_stop = trip_classification.generate_confusion_table(cm_mm_stop, class_names_mm_stop)

print("Combined table with differences")
cm_combined_mm_stop_df = trip_classification.generate_combined_difference_table_all(cm_mm_stop, cm_mm, df_cm_mm_stop, class_names_mm_stop, df_cm_mm)
#print(cm_combined_mm_df)

print("\n")
latex_table_diff_mm_stop = cm_combined_mm_stop_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop)

Macro F1 simple : 0.7559277337218553

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.78      0.81       720
         BUS       0.69      0.53      0.60       489
         CAR       0.93      0.92      0.93      4717
      SUBWAY       0.76      0.58      0.66       733
       TRAIN       0.86      0.74      0.79       462
     TRAMWAY       0.69      0.59      0.64        98
     WALKING       0.81      0.92      0.87      3418

    accuracy                           0.86     10637
   macro avg       0.80      0.72      0.76     10637
weighted avg       0.86      0.86      0.86     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 565 (36) & 5 (-9) & 41 (-17) & 12 (9) & 1 (1) & 1 (0) & 95 (-20) & 720 (0) & 78.472 (5.0) \\
BUS & 10 (

### 250m

In [20]:
temp = df_stop_proximity_250m.copy()
temp = temp.drop(columns="MAIN_MODE")

df_mm_stop_250m = df_merged_motion.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop_250m, y_pred_mm_stop_250m, y_test_mm_stop_250m, class_names_mm_stop_250m = trip_classification.run_xgb_classifier(df_mm_stop_250m)

f1_mm_stop_250m = f1_score(y_test_mm_stop_250m, y_pred_mm_stop_250m, average="macro")
print("Macro F1 simple :", f1_mm_stop_250m)

print("\nClassification report for simple:")
print(classification_report(y_test_mm_stop_250m, y_pred_mm_stop_250m, target_names=class_names_mm_stop_250m))

# Table generation:
cm_mm_stop_250m = metrics.confusion_matrix(y_test_mm_stop_250m, y_pred_mm_stop_250m)
df_cm_mm_stop_250m = trip_classification.generate_confusion_table(cm_mm_stop_250m, class_names_mm_stop_250m)

print("Combined table with differences")
cm_combined_mm_stop_df_250m = trip_classification.generate_combined_difference_table_all(cm_mm_stop_250m, cm_mm, df_cm_mm_stop_250m, class_names_mm_stop_250m, df_cm_mm)
#print(cm_combined_mm_df)

print("\n")
latex_table_diff_mm_stop_250m = cm_combined_mm_stop_df_250m.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop_250m)

Macro F1 simple : 0.7553857833808347

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.78      0.81       720
         BUS       0.70      0.53      0.60       489
         CAR       0.93      0.92      0.92      4717
      SUBWAY       0.76      0.59      0.66       733
       TRAIN       0.85      0.71      0.78       462
     TRAMWAY       0.75      0.58      0.66        98
     WALKING       0.81      0.92      0.86      3418

    accuracy                           0.86     10637
   macro avg       0.80      0.72      0.76     10637
weighted avg       0.86      0.86      0.85     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 561 (32) & 5 (-9) & 42 (-16) & 7 (4) & 1 (1) & 0 (-1) & 104 (-11) & 720 (0) & 77.917 (4.444) \\
BUS & 7

# Stop Proximity + Optimal Percentile

In [21]:
temp = df_stop_proximity_optimal_percentiles.copy()
temp = temp.drop(columns="MAIN_MODE")

df_mm_stop_opt = df_mm_stop.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop_opt, y_pred_mm_stop_opt, y_test_mm_stop_opt, class_names_mm_stop_opt = trip_classification.run_xgb_classifier(df_mm_stop_opt)

f1_mm_stop_opt = f1_score(y_test_mm_stop_opt, y_pred_mm_stop_opt, average="macro")
print("Macro F1 simple :", f1_mm_stop_opt)

print("\nClassification report for simple:")
print(classification_report(y_test_mm_stop_opt, y_pred_mm_stop_opt, target_names=class_names_mm_stop_opt))

# Table generation:
cm_mm_stop_opt = metrics.confusion_matrix(y_test_mm_stop_opt, y_pred_mm_stop_opt)
df_cm_mm_stop_opt = trip_classification.generate_confusion_table(cm_mm_stop_opt, class_names_mm_stop_opt)

print("Combined table with differences")
cm_combined_mm_stop_opt_df = trip_classification.generate_combined_difference_table_all(cm_mm_stop_opt, cm_mm_stop, df_cm_mm_stop_opt, class_names_mm_stop_opt, df_cm_mm_stop)

print("\n")
latex_table_diff_mm_stop_opt = cm_combined_mm_stop_opt_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop_opt)

Macro F1 simple : 0.7528153324395225

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.83      0.79      0.81       720
         BUS       0.70      0.52      0.60       489
         CAR       0.93      0.92      0.93      4717
      SUBWAY       0.76      0.59      0.66       733
       TRAIN       0.85      0.74      0.79       462
     TRAMWAY       0.71      0.54      0.61        98
     WALKING       0.82      0.92      0.87      3418

    accuracy                           0.86     10637
   macro avg       0.80      0.72      0.75     10637
weighted avg       0.86      0.86      0.86     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 572 (7) & 4 (-1) & 41 (0) & 7 (-5) & 1 (0) & 0 (-1) & 95 (0) & 720 (0) & 79.444 (0.972) \\
BUS & 13 (3)

### 250m

In [22]:
temp = df_stop_proximity_optimal_percentiles_250m.copy()
temp = temp.drop(columns="MAIN_MODE")

df_mm_stop_opt_250m = df_mm_stop_250m.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop_opt_250m, y_pred_mm_stop_opt_250m, y_test_mm_stop_opt_250m, class_names_mm_stop_opt_250m = trip_classification.run_xgb_classifier(df_mm_stop_opt_250m)

f1_mm_stop_opt_250m = f1_score(y_test_mm_stop_opt_250m, y_pred_mm_stop_opt_250m, average="macro")
print("Macro F1 simple :", f1_mm_stop_opt_250m)

print("\nClassification report for simple:")
print(classification_report(y_test_mm_stop_opt_250m, y_pred_mm_stop_opt_250m, target_names=class_names_mm_stop_opt_250m))

# Table generation:
cm_mm_stop_opt_250m = metrics.confusion_matrix(y_test_mm_stop_opt_250m, y_pred_mm_stop_opt_250m)
df_cm_mm_stop_opt_250m = trip_classification.generate_confusion_table(cm_mm_stop_opt_250m, class_names_mm_stop_opt_250m)

print("Combined table with differences")
cm_combined_mm_stop_opt_df_250m = trip_classification.generate_combined_difference_table_all(cm_mm_stop_opt_250m, cm_mm_stop_250m, df_cm_mm_stop_opt_250m, class_names_mm_stop_opt_250m, df_cm_mm_stop_250m)

print("\n")
latex_table_diff_mm_stop_opt_250m = cm_combined_mm_stop_opt_df_250m.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop_opt_250m)

Macro F1 simple : 0.758105677484935

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.79      0.81       720
         BUS       0.71      0.55      0.62       489
         CAR       0.94      0.92      0.93      4717
      SUBWAY       0.73      0.59      0.66       733
       TRAIN       0.85      0.72      0.78       462
     TRAMWAY       0.72      0.58      0.64        98
     WALKING       0.81      0.93      0.87      3418

    accuracy                           0.86     10637
   macro avg       0.80      0.73      0.76     10637
weighted avg       0.86      0.86      0.86     10637

Combined table with differences


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 569 (8) & 5 (0) & 38 (-4) & 10 (3) & 0 (-1) & 0 (0) & 98 (-6) & 720 (0) & 79.028 (1.111) \\
BUS & 7 (0) 

### Hybrid: 1000m Stop Proximity + 250m Optimal Percentiles

In [23]:
# Hybrid: 1000m stop proximity features + 250m optimal percentile features
temp = df_stop_proximity_optimal_percentiles_250m.copy()
temp = temp.drop(columns="MAIN_MODE")

# Use df_mm_stop (which has 1000m stop proximity) and merge 250m optimal percentiles
df_mm_stop_hybrid = df_mm_stop.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop_hybrid, y_pred_mm_stop_hybrid, y_test_mm_stop_hybrid, class_names_mm_stop_hybrid = trip_classification.run_xgb_classifier(df_mm_stop_hybrid)

f1_mm_stop_hybrid = f1_score(y_test_mm_stop_hybrid, y_pred_mm_stop_hybrid, average="macro")
print("Macro F1 (1000m stops + 250m optimal percentiles):", f1_mm_stop_hybrid)

print("\nClassification report:")
print(classification_report(y_test_mm_stop_hybrid, y_pred_mm_stop_hybrid, target_names=class_names_mm_stop_hybrid))

# Table generation:
cm_mm_stop_hybrid = metrics.confusion_matrix(y_test_mm_stop_hybrid, y_pred_mm_stop_hybrid)
df_cm_mm_stop_hybrid = trip_classification.generate_confusion_table(cm_mm_stop_hybrid, class_names_mm_stop_hybrid)

print("Combined table with differences (vs 1000m stop proximity)")
cm_combined_mm_stop_hybrid_df = trip_classification.generate_combined_difference_table_all(cm_mm_stop_hybrid, cm_mm_stop, df_cm_mm_stop_hybrid, class_names_mm_stop_hybrid, df_cm_mm_stop)

print("\n")
latex_table_diff_mm_stop_hybrid = cm_combined_mm_stop_hybrid_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop_hybrid)

Macro F1 (1000m stops + 250m optimal percentiles): 0.7648834864942801

Classification report:
              precision    recall  f1-score   support

        BIKE       0.84      0.79      0.81       720
         BUS       0.72      0.57      0.64       489
         CAR       0.94      0.92      0.93      4717
      SUBWAY       0.76      0.59      0.66       733
       TRAIN       0.87      0.73      0.80       462
     TRAMWAY       0.75      0.57      0.65        98
     WALKING       0.81      0.92      0.86      3418

    accuracy                           0.86     10637
   macro avg       0.81      0.73      0.76     10637
weighted avg       0.86      0.86      0.86     10637

Combined table with differences (vs 1000m stop proximity)


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 566 (1) & 6 (1) & 42 (1) & 7 (-5) & 0 (-1) & 0 (-1) & 9

### Hybrid: 250m Stop Proximity + 1000m Optimal Percentiles

In [24]:
# Hybrid: 250m stop proximity features + 1000m optimal percentile features
temp = df_stop_proximity_optimal_percentiles.copy()
temp = temp.drop(columns="MAIN_MODE")

# Use df_mm_stop_250m (which has 250m stop proximity) and merge 1000m optimal percentiles
df_mm_stop_hybrid_inv = df_mm_stop_250m.merge(temp, on='TRIP_KEY')

# -------

xgb_mm_stop_hybrid_inv, y_pred_mm_stop_hybrid_inv, y_test_mm_stop_hybrid_inv, class_names_mm_stop_hybrid_inv = trip_classification.run_xgb_classifier(df_mm_stop_hybrid_inv)

f1_mm_stop_hybrid_inv = f1_score(y_test_mm_stop_hybrid_inv, y_pred_mm_stop_hybrid_inv, average="macro")
print("Macro F1 (250m stops + 1000m optimal percentiles):", f1_mm_stop_hybrid_inv)

print("\nClassification report:")
print(classification_report(y_test_mm_stop_hybrid_inv, y_pred_mm_stop_hybrid_inv, target_names=class_names_mm_stop_hybrid_inv))

# Table generation:
cm_mm_stop_hybrid_inv = metrics.confusion_matrix(y_test_mm_stop_hybrid_inv, y_pred_mm_stop_hybrid_inv)
df_cm_mm_stop_hybrid_inv = trip_classification.generate_confusion_table(cm_mm_stop_hybrid_inv, class_names_mm_stop_hybrid_inv)

print("Combined table with differences (vs 250m stop proximity)")
cm_combined_mm_stop_hybrid_inv_df = trip_classification.generate_combined_difference_table_all(cm_mm_stop_hybrid_inv, cm_mm_stop_250m, df_cm_mm_stop_hybrid_inv, class_names_mm_stop_hybrid_inv, df_cm_mm_stop_250m)

print("\n")
latex_table_diff_mm_stop_hybrid_inv = cm_combined_mm_stop_hybrid_inv_df.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_mm_stop_hybrid_inv)

Macro F1 (250m stops + 1000m optimal percentiles): 0.748245837632081

Classification report:
              precision    recall  f1-score   support

        BIKE       0.84      0.78      0.81       720
         BUS       0.67      0.53      0.59       489
         CAR       0.93      0.92      0.92      4717
      SUBWAY       0.76      0.57      0.65       733
       TRAIN       0.86      0.73      0.79       462
     TRAMWAY       0.71      0.53      0.61        98
     WALKING       0.81      0.92      0.86      3418

    accuracy                           0.86     10637
   macro avg       0.80      0.71      0.75     10637
weighted avg       0.86      0.86      0.85     10637

Combined table with differences (vs 250m stop proximity)


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 562 (1) & 8 (3) & 40 (-2) & 9 (2) & 1 (0) & 0 (0) & 100 (

## Final classifier with route-match features

In [26]:
df_mapmatched = pd.read_csv("/home/s232534/scripts/map_matching/map_matching_interpolated_percentages.csv") 
df_mapmatched = df_mapmatched.drop(columns=['MAIN_MODE', 'MULTI_MODAL'])

df_final = df_mm_stop_opt.merge(df_mapmatched, on='TRIP_KEY')

# -------

xgb_final, y_pred_final, y_test_final, class_names_final = trip_classification.run_xgb_classifier(df_final)

f1_final = f1_score(y_test_final, y_pred_final, average="macro")
print("Macro F1 (250m stops + 1000m optimal percentiles):", f1_final)

print("\nClassification report:")
print(classification_report(y_test_final, y_pred_final, target_names=class_names_final))

# Table generation:
cm_final = metrics.confusion_matrix(y_test_final, y_pred_final)
df_cm_final = trip_classification.generate_confusion_table(cm_final, class_names_final)

print("Combined table with differences (vs 250m stop proximity)")
cm_combined_final = trip_classification.generate_combined_difference_table_all(cm_final, cm_mm_stop_hybrid_inv, df_cm_final, class_names_final, df_cm_mm_stop_hybrid_inv)

print("\n")
latex_table_diff_final = cm_combined_final.fillna("-").to_latex(index=True, float_format="%.3f")
print(latex_table_diff_final)

Macro F1 (250m stops + 1000m optimal percentiles): 0.7814656412853392

Classification report:
              precision    recall  f1-score   support

        BIKE       0.85      0.79      0.82       720
         BUS       0.71      0.57      0.63       489
         CAR       0.94      0.92      0.93      4717
      SUBWAY       0.78      0.58      0.66       733
       TRAIN       0.87      0.78      0.82       462
     TRAMWAY       0.78      0.69      0.74        98
     WALKING       0.81      0.92      0.87      3418

    accuracy                           0.87     10637
   macro avg       0.82      0.75      0.78     10637
weighted avg       0.87      0.87      0.86     10637

Combined table with differences (vs 250m stop proximity)


\begin{tabular}{llllllllll}
\toprule
Predicted class & BIKE & BUS & CAR & SUBWAY & TRAIN & TRAMWAY & WALKING & Row Sum & Row Recall \\
Actual class &  &  &  &  &  &  &  &  &  \\
\midrule
BIKE & 567 (5) & 10 (2) & 47 (7) & 8 (-1) & 0 (-1) & 0 (0) & 88

In [28]:
# save the model:
xgb_final.save_model("/home/s232534/scripts/xgb_final_classifier.json")

### Feature permutations

In [27]:
X_train, X_test, _, _, _ = trip_classification.encode_and_split(df_final)
classifier_feature_cols = xgb_final.feature_names_in_

X_test_sorted = X_test[classifier_feature_cols]

r = permutation_importance(xgb_final, X_test_sorted, y_test_final,
                           n_repeats=30,
                           random_state=0,
                           n_jobs=cpu_count())

print(f"Classifier score: {xgb_final.score(X_test_sorted, y_test_final)}")
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{classifier_feature_cols[i]:<8} "
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")

Classifier score: 0.8665977249224406
SPEED_P95 0.084 +/- 0.002
SPEED_STD 0.054 +/- 0.002
train_match_pct 0.023 +/- 0.001
SPEED_AVG 0.020 +/- 0.001
TOTAL_TIME 0.019 +/- 0.002
bus_match_pct 0.016 +/- 0.001
GPS_POINTS 0.014 +/- 0.001
SPEED_MED 0.014 +/- 0.001
DIST_TOTAL_WGS84 0.009 +/- 0.001
STOPS_MINDIST_AVG 0.008 +/- 0.001
STOPS_CLOSEST_RAIL_P20 0.007 +/- 0.001
MIN_DIST_METRO 0.007 +/- 0.001
ACC_STD  0.006 +/- 0.001
DURATION_METRO 0.006 +/- 0.001
ACC_P95  0.006 +/- 0.001
tram_match_pct 0.005 +/- 0.001
STOPS_NUM_MAX 0.005 +/- 0.001
JERK_STD 0.005 +/- 0.001
STOPS_CLOSEST_METRO_P95 0.005 +/- 0.001
MED_PT_DIST_WGS84 0.004 +/- 0.001
subway_match_pct 0.004 +/- 0.001
ACC_AVG  0.003 +/- 0.001
STOPS_NUM_AVG 0.003 +/- 0.001
AVG_PT_DIST_WGS84 0.003 +/- 0.001
STOPS_NUM_MIN 0.002 +/- 0.001
DURATION_BUS 0.002 +/- 0.001
STOPS_CLOSEST_TRAM_P15 0.002 +/- 0.001
MIN_DIST_RAIL 0.002 +/- 0.001
BR_MED_WGS84 0.002 +/- 0.001
MIN_DIST_BUS 0.002 +/- 0.001
JERK_MED 0.002 +/- 0.001
BC_MED_WGS84 0.002 +/- 0.001
STO